# Fine-Tune Evidence-Aware RAG Synthesizer

This notebook uses QLoRA (4-bit quantization) to fine-tune `Qwen1.5-0.5B-Chat` to act as an advanced, hallucination-free RAG Synthesizer. It is designed specifically to fit on a 4GB VRAM GPU (like a GTX 1650 Ti).

In [ ]:
# Install required packages into this specific notebook kernel
%pip install datasets trl peft bitsandbytes

In [1]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer,  BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

os.environ['CUDA_MODULE_LOADING'] = 'EAGER'
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


CUDA: NVIDIA GeForce GTX 1650 Ti | Memory: 4.29 GB


### Load Dataset

In [2]:
# Load our custom generated dataset
dataset = load_dataset('json', data_files={'train': 'D:/RES_cove/adaptive-evidence-rag/data/finetune/train.jsonl', 'val': 'D:/RES_cove/adaptive-evidence-rag/data/finetune/val.jsonl'})
print(dataset)

def formatting_prompts_func(example):
    if isinstance(example['instruction'], str):
        # Return a single string for a single un-batched example
        return f"{example['instruction']}\n{example['output']}<|im_end|>"
    # Return a list of strings for a batch
    output_texts = []
    for i in range(len(example['instruction'])):
        text = f"{example['instruction'][i]}\n{example['output'][i]}<|im_end|>"
        output_texts.append(text)
    return output_texts


DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1800
    })
    val: Dataset({
        features: ['instruction', 'output'],
        num_rows: 200
    })
})


### Load Model & Quantization

In [4]:
model_name = "Qwen/Qwen1.5-0.5B-Chat"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Loading 4-bit Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)


Loading Tokenizer...
Loading 4-bit Model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

### LoRA Configuration

In [5]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 3,784,704 || all params: 467,772,416 || trainable%: 0.8091


### Training Setup

In [6]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="../models/qwen-evidence-rag-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=500,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    report_to="none",
    max_length=1024
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    formatting_func=formatting_prompts_func,
    processing_class=tokenizer,
    args=training_args,
)

# Clear cache before training
torch.cuda.empty_cache()



[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
C:\Users\dynos\AppData\Local\Temp\ipykernel_8776\1538972892.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


In [7]:
print("Starting fine-tuning...")
trainer.train()

# Save the final LoRA adapters
trainer.model.save_pretrained("../models/qwen-evidence-rag-lora-final")
print("Training complete and model saved!")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting fine-tuning...


RuntimeError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'